<a href="https://colab.research.google.com/github/hle369518-dotcom/TriTueNhantao/blob/main/LHHuy_AKT_Astar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#AKT
import copy
from heapq import heappush, heappop

# Giả sử puzzle(n=3)
n = 3

# 4 vị trí dịch chuyển tương ứng bottom, left, top, right
rows = [ 1, 0, -1, 0 ]
cols = [ 0, -1, 0, 1 ]

# Tạo một lớp hàng đợi
class priorityQueue:
    def __init__(self):
        self.heap = []

    # Inserting a new key 'key'
    def push(self, key):
        heappush(self.heap, key)

    # funct to remove the element that is min from the Priority Queue
    def pop(self):
        return heappop(self.heap)

    # funct to check if the Queue is empty or not
    def empty(self):
        if not self.heap:
            return True
        else:
            return False

# structure of the node
class nodes:
    def __init__(self, parent, mats, empty_tile_pos, costs, levels):
        self.parent = parent
        # Useful for Storing the matrix
        self.mats = mats
        self.empty_tile_pos = empty_tile_pos
        self.costs = costs
        self.levels = levels

    def __lt__(self, nxt):
        return self.costs < nxt.costs

def calculateCosts(mats, final) -> int:
    count = 0
    for i in range(n):
        for j in range(n):
            if ((mats[i][j]) and (mats[i][j] != final[i][j])):
                count += 1
    return count

def newNodes(mats, empty_tile_pos, new_empty_tile_pos, levels, parent, final) -> nodes:
    # Copying data from the parent matrixes to the present matrixes
    new_mats = copy.deepcopy(mats)

    # Moving the tile by 1 position
    x1 = empty_tile_pos[0]
    y1 = empty_tile_pos[1]
    x2 = new_empty_tile_pos[0]
    y2 = new_empty_tile_pos[1]

    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    # Setting the no. of misplaced tiles
    costs = calculateCosts(new_mats, final)

    new_nodes = nodes(parent, new_mats, new_empty_tile_pos, costs, levels)
    return new_nodes

# func to print the N by N matrix
def printMatsrix(mats):
    for i in range(n):
        for j in range(n):
            print("%d " % (mats[i][j]), end = " ")
        print()

# func to know if (x, y) is a valid or invalid matrix coordinates
def isSafe(x, y):
    return x >= 0 and x < n and y >= 0 and y < n

# Printing the path from the root node to the final node
def printPath(root):
    if root == None:
        return

    printPath(root.parent)
    printMatsrix(root.mats)
    print()

def solve(initial, empty_tile_pos, final):
    pq = priorityQueue()

    # Creating the root node
    costs = calculateCosts(initial, final)
    root = nodes(None, initial, empty_tile_pos, costs, 0)

    # Adding root to the list of live nodes
    pq.push(root)

    while not pq.empty():
        minimum = pq.pop()

        if minimum.costs == 0:
            printPath(minimum)
            return

        # Generating all feasible children
        for i in range(4):
            new_tile_pos = [
                minimum.empty_tile_pos[0] + rows[i],
                minimum.empty_tile_pos[1] + cols[i],
            ]

            if isSafe(new_tile_pos[0], new_tile_pos[1]):
                # Creating a child node
                child = newNodes(minimum.mats,
                                 minimum.empty_tile_pos,
                                 new_tile_pos,
                                 minimum.levels + 1,
                                 minimum,
                                 final)
                # Adding the child to the list of live nodes
                pq.push(child)

# Chạy chương trình
initial = [ [ 1, 2, 3 ],
            [ 5, 6, 0 ],
            [ 7, 8, 4 ] ]

final = [ [ 1, 2, 3 ],
          [ 5, 8, 6 ],
          [ 0, 7, 4 ] ]

empty_tile_pos = [1, 2]
solve(initial, empty_tile_pos, final)


1  2  3  
5  6  0  
7  8  4  

1  2  3  
5  0  6  
7  8  4  

1  2  3  
5  8  6  
7  0  4  

1  2  3  
5  8  6  
0  7  4  



In [4]:
#A*
from collections import deque

class Graph:
    def __init__(self, adjac_lis):
        self.adjac_lis = adjac_lis

    def get_neighbors(self, v):
        return self.adjac_lis[v]

    # This is heuristic function which is having equal values for all nodes
    def h(self, n):
        H = {
            'A': 1,
            'B': 1,
            'C': 1,
            'D': 1
        }
        return H[n]

    def a_star_algorithm(self, start, stop):
        open_lst = set([start])
        closed_lst = set([])

        # poo stores distance from start to the node
        poo = {}
        poo[start] = 0

        # par contains an adjac mapping of all nodes
        par = {}
        par[start] = start

        while len(open_lst) > 0:
            n = None

            # it will find a node with the lowest value of f()
            for v in open_lst:
                if n == None or poo[v] + self.h(v) < poo[n] + self.h(n):
                    n = v;

            if n == None:
                print('Path does not exist!')
                return None

            # if the current node is the stop then we start again from start
            if n == stop:
                reconst_path = []

                while par[n] != n:
                    reconst_path.append(n)
                    n = par[n]

                reconst_path.append(start)
                reconst_path.reverse()

                print('Path found: {}'.format(reconst_path))
                return reconst_path

            # for all the neighbors of the current node do
            for (m, weight) in self.get_neighbors(n):
                # if the current node is not present in both open_lst and closed_lst
                # add it to open_lst and note n as it's par
                if m not in open_lst and m not in closed_lst:
                    open_lst.add(m)
                    par[m] = n
                    poo[m] = poo[n] + weight

                else:
                    if poo[m] > poo[n] + weight:
                        poo[m] = poo[n] + weight
                        par[m] = n

                    if m in closed_lst:
                        closed_lst.remove(m)
                        open_lst.add(m)

            open_lst.remove(n)
            closed_lst.add(n)

        print('Path does not exist!')
        return None

# Chạy thử thuật toán A*
adjac_lis = {
    'A': [('B', 1), ('C', 3), ('D', 7)],
    'B': [('C', 1), ('D', 5)],
    'C': [('D', 12)]
}

graph1 = Graph(adjac_lis)
graph1.a_star_algorithm('A', 'C')

Path found: ['A', 'B', 'C']


['A', 'B', 'C']